# Using LLMs with Images via an OpenAI-Compatible API

This notebook shows how to send images to a vision-capable language model through any endpoint that speaks the OpenAI Chat Completions protocol.

We will use the NRP hosted LLMs, but the same code works with the OpenAI API, local models served with Ollama, and any other providers or locally running applications that provide services through an Open-AI compatible endpoint.  

Because they all share a common format, switching between providers/servers is usually just a change of `base_url`, `api_key`, and `model`.

## Set-Up

We use the official `openai` Python package as a generic OpenAI-compatible client. `httpx` comes along for the image-fetching helper later.

Edit the three values below to point at your provider. The API key can be a dummy string for most local servers (they don't check it), but the field still has to be present.

In [ ]:
from openai import OpenAI
import keys

# --- Edit these lines for your setup --------------------------------

# BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
BASE_URL = "https://ellm.nrp-nautilus.io/v1"

# API_KEY  = os.environ.get("OPENAI_API_KEY", "sk-...")   # dummy is fine for local servers
API_KEY = keys.NRP_TOK

# MODEL    = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
MODEL = 'qwen3-small'

# For NRP, we also set up the cache salt
NRP_CACHE_SALT = keys.NRP_CACHE_SALT

# --------------------------------------------------------------------------

llm_client = OpenAI(base_url = BASE_URL,
                    api_key = API_KEY)

print(f"Talking to {BASE_URL} using model '{MODEL}'")

## The simplest case: an image by URL

A vision request looks like an ordinary chat request, except the user message's `content` is a list of parts instead of a plain string. Each part is either `{"type": "text", ...}` or `{"type": "image_url", ...}`.

Note: passing a remote URL requires that the model server be able to reach that URL. Many local servers can't, so for those use the base64 method in the next section.

In [ ]:
# For NRP, this will give a 403 Forbidden error, due to access restrictions

# IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/85/Orange_tabby_cat_sitting_on_fallen_leaves-Hisashi-01.jpg/500px-Orange_tabby_cat_sitting_on_fallen_leaves-Hisashi-01.jpg"

# response = llm_client.chat.completions.create(
#     model=MODEL,
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                 {"type": "text", "text": "Describe this image in one sentence."},
#                 {"type": "image_url", "image_url": {"url": IMAGE_URL}},
#             ],
#         }
#     ],
#     extra_body={"cache_salt": NRP_CACHE_SALT}
# )

# print(response.choices[0].message.content)

## Local images: base64 data URIs

The most portable approach is to read the file, base64-encode it, and embed it as a `data:` URI. This works everywhere because the bytes travel inside the request — the server never has to fetch anything.

In [ ]:
import base64
import mimetypes
from pathlib import Path
from io import BytesIO
from skimage import data, io

def image_to_data_uri(path: str) -> str:
    """Read a local image file and return a base64 data URI."""
    
    path = Path(path)

    mime, _ = mimetypes.guess_type(path)
    if mime is None:
        mime = "image/jpeg"  # sensible fallback
    
    b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    
    return f"data:{mime};base64,{b64}"

def skimage_to_data_uri(d):
    """Read scikit-image data "d" and return a base64 data URI."""

    buffer = BytesIO()
    io.imsave(buffer, d, format='jpeg')

    b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')   

    return f"data:image/jpeg;base64,{b64}"

In [ ]:
IMAGE_PATH = 'grogu_clarinet.jpg'

# Preview
from IPython.display import Image as IPyImage
IPyImage(filename=IMAGE_PATH, width=320)

In [ ]:
grogu_data = image_to_data_uri(IMAGE_PATH)
# grogu_data[:40]+'...'
grogu_data

In [ ]:
data_uri = image_to_data_uri(IMAGE_PATH)

response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What is the main subject, and what is it doing?"},
                {"type": "image_url", "image_url": {"url": data_uri}},
            ],
        }
    ],
)

print(response.choices[0].message.content)

## Controlling resolution with `detail`

OpenAI (and several compatible servers) accept a `detail` field on the image:
`"low"`, `"high"`, or `"auto"`. `"low"` downsamples the image to a small fixed
size — cheaper and faster, at the cost of fine detail. Servers that don't
understand the field generally just ignore it, so it's safe to include.

In [ ]:
response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Roughly how many objects are in this picture?"},
                {
                    "type": "image_url",
                    "image_url": {"url": data_uri, "detail": "low"},
                },
            ],
        }
    ],
)

print(response.choices[0].message.content)

## Multiple images in one message

You can include several `image_url` parts in the same message — useful for
comparison, before/after, or "which of these..." style questions. Order is
preserved, so you can refer to "the first image" and "the second image" in
your prompt.

In [ ]:
data_uri_2 = skimage_to_data_uri(data.chelsea())
data_uri_2[:40]

In [ ]:
response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Compare the two animals. What species is each, and how do they differ?"},
                {"type": "image_url", "image_url": {"url": data_uri}},
                {"type": "image_url", "image_url": {"url": data_uri_2}},
            ],
        }
    ],
)

print(response.choices[0].message.content)

## Streaming the response

For longer descriptions, stream tokens as they arrive instead of waiting for the whole reply. Set `stream=True` and iterate over the chunks.

In [ ]:
stream = llm_client.chat.completions.create(
    model=MODEL,
    stream=True,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Write a vivid, detailed paragraph describing this scene."},
                {"type": "image_url", "image_url": {"url": data_uri}},
            ],
        }
    ],
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

## Multi-turn conversations about an image

The image only needs to be sent once. On follow-up turns you append the assistant's previous reply and your new question as ordinary text messages — the model retains the visual context from earlier in the conversation.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What color is the main subject?"},
            {"type": "image_url", "image_url": {"url": data_uri}},
        ],
    }
]

# First turn
first = llm_client.chat.completions.create(model=MODEL, messages=messages)
answer = first.choices[0].message.content
print("Q1: What color is the main subject?")
print("A1:", answer, "\n")

# Append the assistant turn + a follow-up (text only — no need to resend the image)
messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "And what is in the background?"})

second = llm_client.chat.completions.create(model=MODEL, messages=messages)
print("Q2: And what is in the background?")
print("A2:", second.choices[0].message.content)

## Getting structured output

A common goal is machine-readable results. Ask for JSON in the prompt and, if
your provider supports it, set `response_format={"type": "json_object"}` to
force valid JSON. Providers that don't support that field will ignore it, so
the prompt instruction is your real guarantee — always parse defensively.

In [ ]:
import json

response = llm_client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},  # ignored by servers that don't support it
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Return ONLY a JSON object with keys: "
                        "subject (string), setting (string), "
                        "dominant_colors (array of strings), "
                        "contains_text (boolean)."
                    ),
                },
                {"type": "image_url", "image_url": {"url": data_uri}},
            ],
        }
    ],
)

raw = response.choices[0].message.content
try:
    parsed = json.loads(raw)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("Model did not return clean JSON. Raw output:\n", raw)

## Without the SDK: a raw HTTP request

The SDK is just sugar over an HTTP POST. If you'd rather not depend on it (or you're porting to another language), here's the same vision call with plain `httpx`. The JSON body is exactly what every OpenAI-compatible server expects.

In [ ]:
import httpx

payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image briefly."},
                {"type": "image_url", "image_url": {"url": data_uri}},
            ],
        }
    ],
}

resp = httpx.post(
    f"{BASE_URL}/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    json=payload,
    timeout=120,
)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])

## Potential Info for Troubleshooting

- **`model does not support images` / 400 errors** — you're pointed at a text-only model. Pick a vision model.
- **Remote URL fails but base64 works** — the server can't reach the internet. Use the base64 data-URI method; it's the most reliable.
- **413 / payload too large** — downscale before encoding. Most vision models see no benefit above ~1024px on the long edge, and it dramatically shrinks the request. Use Pillow: `img.thumbnail((1024, 1024))` then re-save.
- **Unsupported `detail` or `response_format`** — these are silently ignored by many local servers; rely on prompt instructions as the real contract.
- **Wrong MIME type** — `image_to_data_uri` guesses from the extension. If your file has no/odd extension, pass the correct `image/png`, `image/jpeg`, etc.
- **Auth errors against a local server** — most ignore the key, but the field must still be a non-empty string.

We've covered the bare essentials: a chat request whose content is a list of text and `image_url` parts. Everything else is the normal chat-completions API.